In [0]:
%sql
-- ============================================================================
-- Pipeline Step: Gold Fact - fact_wikipedia_edits
-- Description: Central fact table tracking edit events linked to dimensional keys.
-- Dependencies: 
--   - dbr_dev.wikimediademo_silver.wikipedia_edits (Ivan)
--   - dim_date, dim_project, dim_editor_type
-- ============================================================================
USE CATALOG dbr_dev;
USE SCHEMA wikimediademo_gold;

CREATE OR REPLACE TABLE dbr_dev.wikimediademo_gold.fact_wikipedia_edits AS
SELECT 
    s.event_id,
    cast(date_format(coalesce(s.event_time, s.silver_ingested_at), 'yyyyMMdd') AS INT) AS date_key,
    
    -- Assign valid project_key, or fallback to the 'unknown' project key
    COALESCE(p.project_key, md5('unknown')) AS project_key,
    
    CASE 
        WHEN s.bot = true THEN 1 
        ELSE 2 
    END AS editor_type_key,
    
    s.user,
    s.title,
    s.event_time,
    s.length_old,
    s.length_new,
    (s.length_new - s.length_old) AS byte_change,
    abs(s.length_new - s.length_old) AS abs_byte_change,
    coalesce(s.minor, false) AS is_minor_edit,
    current_timestamp() AS gold_ingested_at
FROM dbr_dev.wikimediademo_silver.silver_wikipedia_edits s
LEFT JOIN dbr_dev.wikimediademo_gold.dim_project p
    ON md5(coalesce(s.wiki, 'unknown')) = p.project_key;


In [0]:
%sql
-- Verification
SELECT * FROM dbr_dev.wikimediademo_gold.fact_wikipedia_edits LIMIT 5;